# Evaluation for LLM Apps — Retrieval Metrics, LLM-as-Judge, and LangSmith

`vector_databases_for_rag` Section 8 already built precision@k, recall@k, and MRR against a small, hand-labeled 4-question eval set. `03_advanced_retrievers` ended on an exercise it deliberately didn't answer: *actually measure* whether `MultiQueryRetriever` and `SelfQueryRetriever` outperform plain search, instead of trusting that "advanced" means "better." This notebook does that — then goes one level further.

Retrieval metrics only answer "did we find the right chunks?" They say nothing about whether the *final generated answer* is any good — a pipeline can retrieve perfectly and still hallucinate, misread the context, or ramble past the actual question. So this notebook covers three layers, each answering a different question:

1. **Retrieval metrics** (precision@k / recall@k / MRR) — did we find the right chunks? Extends `vector_databases_for_rag` §8's exact functions to a bigger, more realistic labeled set, run against plain search, `MultiQueryRetriever`, and `SelfQueryRetriever` side by side.
2. **LLM-as-judge** — given the right chunks, is the generated answer actually good? A second LLM call scores faithfulness and relevance, with structured output (ties back to `Build_Smarter_AI_Apps` §5's `with_structured_output`).
3. **LangSmith** — tracing (see exactly what happened on any call, not just what you chose to `print()`), a real hosted dataset (replacing the Python list), and `evaluate()` as a repeatable regression test — same dataset, same evaluators, comparable "experiments" over time.

**On LangSmith Engine:** the LangSmith docs also describe an "Engine" feature — an autonomous agent that watches a *live production* agent's traces every 6 hours, diagnoses recurring issues, and opens PRs with fixes. It's a real, current LangSmith capability, but it's a closed, paid (LCU-billed), cloud-managed product feature with no code to write or run — and it needs real production traffic to have anything to diagnose. That breaks the "actually run this" pattern this whole curriculum has followed, so it isn't built hands-on here. Worth knowing it exists as the natural next step once both evaluation and tracing (this notebook) are in place — it'll get a proper mention in Module 08's production-monitoring section instead.

**On execution:** same transparency rule as `03_advanced_retrievers` — every cell that doesn't need your Anthropic *or* LangSmith key was actually run while building this, with real saved output. Cells needing either key are marked **⚠️ needs your API key(s)** and left for you to run.

## Setup

In [1]:
# pip install -U langchain langchain-anthropic langchain-classic langchain-chroma langchain-huggingface langgraph langsmith pydantic

import os
import getpass

# Check the environment first -- if you've already exported ANTHROPIC_API_KEY in
# your shell, this cell never prompts at all. getpass is only the fallback, and
# it never echoes what you type or leaves the value sitting in the notebook file.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
print("Anthropic key ready.")

# LangSmith is optional here -- Sections 1-4 work with zero LangSmith setup at all.
# Only Section 5 needs this. Leave the prompt blank to skip it cleanly.
if not os.environ.get("LANGSMITH_API_KEY"):
    ls_key = getpass.getpass("Enter your LangSmith API key (leave blank to skip Section 5): ")
    if ls_key:
        os.environ["LANGSMITH_API_KEY"] = ls_key

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"  # the exact env var name the installed langsmith SDK checks -- verified directly, not assumed
    os.environ.setdefault("LANGSMITH_PROJECT", "module-04-eval")
    print("LangSmith tracing enabled.")
else:
    print("No LangSmith key -- Sections 1-4 still work fully, Section 5 will just show the code.")

from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


def get_llm(max_tokens=512, temperature=0):
    """Same factory pattern as 01/02/03 -- lets each call site pick its own
    max_tokens/temperature without constructing a new client by hand each time."""
    return ChatAnthropic(model="claude-haiku-4-5", max_tokens=max_tokens, temperature=temperature)


print("Imports OK")

Anthropic key ready.
LangSmith tracing enabled.


/Users/Sarah/venvs/upskill2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Rebuilding the knowledge base and retrievers from `03_advanced_retrievers`

Self-contained, same as every notebook in this series — the exact 10-policy set from 03, with one addition: an explicit `doc_id` in metadata (same pattern `vector_databases_for_rag` §8 used), since a labeled eval set needs a stable identifier to check against, and Chroma's own internal IDs aren't something you'd want to hand-write into a labeled test set. `department`/`year`/`priority` are unchanged from 03, still there for `SelfQueryRetriever`.

`policies` is a list of 3-element tuples: (doc_id_string, document_text_string, metadata_dict). Looking at one entry:


("hr_vacation",              # 1. doc_id -- a plain string, NOT inside a dict
 "Vacation Policy: Full-time employees accrue 15 ...",       # 2. the document text
 {"department": "HR", "year": 2023, "priority": "standard"}) # 3. metadata dict -- doc_id is NOT in here

In [2]:
policies = [
    # Each tuple: (doc_id, page_content_text, metadata_dict).
    # doc_id is the label vocabulary eval_set will reference below --
    # e.g. "hr_vacation" here is the exact string that shows up in
    # eval_set's {"hr_vacation"} ground-truth sets further down.
    ("hr_remote", "Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. "
     "Requests must be submitted at least 48 hours in advance through the HR portal.",
     {"department": "HR", "year": 2024, "priority": "standard"}),
    ("hr_vacation", "Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly. "
     "Unused vacation days roll over up to a maximum of 5 days into the following year.",
     {"department": "HR", "year": 2023, "priority": "standard"}),
    ("hr_parental", "Parental Leave Policy: Employees are eligible for 12 weeks of paid leave following the birth, "
     "adoption, or fostering of a child, regardless of gender.",
     {"department": "HR", "year": 2025, "priority": "standard"}),
    ("it_mobile", "Mobile Phone Policy: Company-issued phones may be used for personal calls within reason. "
     "Employees must report a lost or stolen device to IT within 24 hours.",
     {"department": "IT", "year": 2023, "priority": "standard"}),
    ("finance_expense", "Expense Policy: Business expenses under $50 do not require pre-approval but must be "
     "submitted with receipts within 30 days. Expenses over $50 require manager sign-off before purchase.",
     {"department": "Finance", "year": 2024, "priority": "standard"}),
    ("it_security", "Data Security Policy: All customer data must be encrypted at rest and in transit. Any "
     "suspected breach must be reported to the security team within 1 hour of discovery.",
     {"department": "IT", "year": 2025, "priority": "critical"}),
    ("it_password", "Password Policy: Passwords must be at least 14 characters and rotated every 90 days. "
     "Multi-factor authentication is mandatory for all systems handling customer data.",
     {"department": "IT", "year": 2025, "priority": "critical"}),
    ("finance_travel", "Travel and Reimbursement Policy: Employees traveling for business must book through the "
     "approved travel portal. Reimbursement requests must be filed within 14 days of return.",
     {"department": "Finance", "year": 2025, "priority": "standard"}),
    ("legal_conduct", "Code of Conduct: Employees must avoid conflicts of interest and disclose any outside "
     "business relationships that could affect their judgment at the company.",
     {"department": "Legal", "year": 2023, "priority": "critical"}),
    ("legal_nda", "Non-Disclosure Policy: Employees must not share confidential company information with third "
     "parties without prior written approval from the Legal department.",
     {"department": "Legal", "year": 2024, "priority": "critical"}),
]

# doc_id becomes part of a metadata dict in the list comprehension that builds the actual Document objects:
# note it's merged INTO the same dict as department/year/priority (via **meta) -- Chroma's
# metadata is just one flat dict per document, it doesn't distinguish "id-like" fields
# from "filter-like" fields. The distinction between them is purely how we use them later.

docs = [
    Document(page_content=text, metadata={"doc_id": doc_id, **meta})
    for doc_id, text, meta in policies
]

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Same embedding model as 03 -- consistency matters here specifically because eval scores
# are only comparable to 03's if the underlying retrieval quality is coming from the same model.

# Chroma.from_documents(..., collection_name=...) is NOT idempotent -- re-running this cell
# in the same kernel session (completely normal while iterating on a notebook) silently
# APPENDS a second copy of every document to the same collection instead of replacing it,
# since there's no persist_directory and nothing here checks for an existing collection.
# Left unguarded, this is exactly what produced an impossible recall@3 > 1.0 further down:
# duplicate copies of the same relevant doc filled multiple slots in top-k, each counted as
# a separate hit. Deleting any existing collection with this name first makes this cell safe
# to re-run any number of times in the same session.
Chroma(collection_name="module_04_eval_policies", embedding_function=embedding_model).delete_collection()
vectorstore = Chroma.from_documents(docs, embedding_model, collection_name="module_04_eval_policies")
print(f"Indexed {len(docs)} documents.")

Indexed 10 documents.


In [8]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.structured_query import Comparator, Operator, Comparison, Operation, StructuredQuery, Visitor


class SimpleChromaTranslator(Visitor):
    """Same hand-written translator from 03 -- langchain-community still isn't
    installed, still isn't getting added for this one class."""

    # Only the operators/comparators Chroma's `where` syntax actually supports --
    # this is what would raise a clear error if the LLM tried e.g. Comparator.LIKE.
    allowed_comparators = (
        Comparator.EQ, Comparator.NE, Comparator.GT, Comparator.GTE,
        Comparator.LT, Comparator.LTE, Comparator.IN, Comparator.NIN,
    )
    allowed_operators = (Operator.AND, Operator.OR)
    _op_map = {Operator.AND: "$and", Operator.OR: "$or"}
    _cmp_map = {
        Comparator.EQ: "$eq", Comparator.NE: "$ne", Comparator.GT: "$gt",
        Comparator.GTE: "$gte", Comparator.LT: "$lt", Comparator.LTE: "$lte",
        Comparator.IN: "$in", Comparator.NIN: "$nin",
    }

    def visit_operation(self, operation):
        # e.g. AND(eq(department, IT), eq(priority, critical)) -> {"$and": [{...}, {...}]}
        return {self._op_map[operation.operator]: [a.accept(self) for a in operation.arguments]}

    def visit_comparison(self, comparison):
        # e.g. eq(department, IT) -> {"department": {"$eq": "IT"}}
        return {comparison.attribute: {self._cmp_map[comparison.comparator]: comparison.value}}

    def visit_structured_query(self, structured_query):
        # Returns (semantic_query_text, kwargs_for_similarity_search) -- the two
        # halves SelfQueryRetriever recombines into one similarity_search call.
        if structured_query.filter is None:
            return structured_query.query, {}
        return structured_query.query, {"filter": structured_query.filter.accept(self)}


# k=3 here (not 2, like 03) -- matches the k used in every evaluate_retrieval() call
# below, so precision@k/recall@k are computed at a consistent k across all three
# strategies and across Sections 3 and 5.
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Plain retriever, wrapped so an LLM rewrites the question into 3 variants first (03 §3).
multi_query_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=get_llm())

# Same vectorstore, but an LLM also extracts a department/year/priority filter
# from the question text before searching (03 §4).
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=get_llm(),
    vectorstore=vectorstore,
    document_contents="A company policy document",
    metadata_field_info=[
        AttributeInfo(name="department", description="HR, IT, Finance, or Legal", type="string"),
        AttributeInfo(name="year", description="Year the policy was last updated", type="integer"),
        AttributeInfo(name="priority", description="standard or critical", type="string"),
    ],
    structured_query_translator=SimpleChromaTranslator(),
    enable_limit=True,
)
print("All three retrievers constructed (construction alone makes no API call).")

All three retrievers constructed (construction alone makes no API call).


## 2. A real labeled eval set

`vector_databases_for_rag` §8 used 4 single-topic questions. This one is deliberately harder: it includes the exact compound question from `03_advanced_retrievers` §2 (the one plain search provably failed on) and a filter-heavy question matching §4's self-query demo, alongside plain single-topic questions — so the metrics below can actually show a difference between the three retrievers, not just confirm they all get the easy cases right.

In [27]:
docs[:2]

[Document(metadata={'doc_id': 'hr_remote', 'department': 'HR', 'year': 2024, 'priority': 'standard'}, page_content='Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. Requests must be submitted at least 48 hours in advance through the HR portal.'),
 Document(metadata={'doc_id': 'hr_vacation', 'department': 'HR', 'year': 2023, 'priority': 'standard'}, page_content='Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly. Unused vacation days roll over up to a maximum of 5 days into the following year.')]

**Q: would these be human-labelled?**

In a real project, yes — `eval_set` below is exactly the kind of thing a human (someone who actually knows the policies, not the model being evaluated) should write, precisely because it's the ground truth everything else gets measured against. If the model that's supposed to be graded also wrote its own answer key, a systematic blind spot in the model just becomes an invisible blind spot in the eval too — the eval would "pass" without ever checking the thing it's meant to check.

Here, *I* wrote `eval_set` by hand while building this notebook — a stand-in for that human labeling step, since there's no real policy team to ask. Two things worth knowing for when you do this on a real project:
- **LLM-assisted labeling is common in practice** (having a model draft candidate labels to speed up the process), but the drafts always get a human spot-check before going into a real eval set — an unreviewed LLM-labeled eval set has the same blind-spot problem as a self-graded one.
- **The labels only need to be as good as your ground truth allows.** If two people would reasonably disagree about which document answers a question, that question is telling you something about the *question*, not a bug in the eval code.

In [3]:
eval_set = [
    # -- Plain single-topic questions: every retriever should nail these. They're the
    #    control group -- if a "better" retriever scored worse here, that's a red flag,
    #    not a sign it's smarter.
    ("How many vacation days do I get per year?", {"hr_vacation"}),
    ("Can I work from home?", {"hr_remote"}),
    ("What happens if I have a baby?", {"hr_parental"}),
    ("Do I need approval for a $30 expense?", {"finance_expense"}),
    ("How often do I need to change my password?", {"it_password"}),
    ("Can I share confidential information with an outside vendor?", {"legal_nda"}),
    ("How do I get reimbursed after a business trip?", {"finance_travel"}),

    # -- Filter-heavy: TWO correct docs, both requiring department=IT AND priority=critical.
    #    This is the one designed to reward SelfQueryRetriever specifically.
    ("What are the critical IT policies?", {"it_security", "it_password"}),

    # -- Compound/multi-facet: THREE correct docs across 3 different departments, the
    #    exact question 03 §2 proved plain search misses one of. Designed to reward
    #    MultiQueryRetriever specifically.
    (
        "What do I need to know about working from home, keeping my laptop secure, "
        "and expensing my home office chair?",
        {"hr_remote", "it_security", "finance_expense"},
    ),
]
print(f"{len(eval_set)} labeled questions.")

9 labeled questions.


## 3. Retrieval metrics — reused verbatim from `vector_databases_for_rag` §8

Same three functions, unchanged — no reason to reinvent precision@k, recall@k, or MRR a second time.

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    # "Of the k things we handed the model, how many were actually relevant?"
    # Structurally penalizes retrieving too MUCH irrelevant stuff.
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    # "Of everything that WAS relevant, how much did we actually surface?"
    # Structurally penalizes missing relevant docs -- this is the one the
    # compound-question and filter-heavy questions above are designed to test.
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / len(relevant_ids)


def reciprocal_rank(retrieved_ids, relevant_ids):
    # 1/rank of the FIRST relevant hit (1.0 if it's ranked #1, 0.5 if #2, 0 if it
    # never appears at all) -- rewards ranking the right answer early, not just
    # including it somewhere in the list.
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0.0


def evaluate_retrieval(retrieve_fn, k=3):
    """retrieve_fn(question) -> list[Document]. Returns per-question rows and means. 
    AKA each row is associated with one question in eval set."""
    rows = []
    for question, relevant in eval_set:
        retrieved_docs = retrieve_fn(question)  # returns a list of Documents
        # This is the line that reads the doc_id metadata field discussed above --
        # everything below only works because doc_id in eval_set's ground-truth
        # sets and doc_id in each Document's metadata use the exact same strings.
        retrieved_ids = [d.metadata["doc_id"] for d in retrieved_docs]
        rows.append({
            "question": question,
            "precision": precision_at_k(retrieved_ids, relevant, k),
            "recall": recall_at_k(retrieved_ids, relevant, k),
            "rr": reciprocal_rank(retrieved_ids, relevant),
        })
    n = len(rows)
    means = {
        "precision": sum(r["precision"] for r in rows) / n,
        "recall": sum(r["recall"] for r in rows) / n,
        "mrr": sum(r["rr"] for r in rows) / n,  # "MRR" = Mean Reciprocal Rank, the average of every question's reciprocal_rank
    }
    return rows, means


print("Metric functions ready.")

Metric functions ready.


### Plain similarity search — no API key needed, run for real

In [6]:
# k=3 was set on purpose: precision@3 tops out at 0.33 for every single-topic question
# below even when retrieval is PERFECT, because there's only 1 truly relevant doc out
# of the 3 returned -- low precision here is expected, not a bug. Watch recall and MRR
# instead for the single-topic rows; precision only gets interesting on the two
# multi-doc questions (critical IT policies, and the compound question) further down.
plain_rows, plain_means = evaluate_retrieval(lambda q: vectorstore.similarity_search(q, k=3), k=3)

for r in plain_rows:
    print(f"P={r['precision']:.2f} R={r['recall']:.2f} RR={r['rr']:.2f}  | {r['question'][:65]}")

print(f"\nMean precision@3: {plain_means['precision']:.3f}")
print(f"Mean recall@3:    {plain_means['recall']:.3f}")
print(f"MRR:              {plain_means['mrr']:.3f}")

P=0.33 R=1.00 RR=1.00  | How many vacation days do I get per year?
P=0.33 R=1.00 RR=1.00  | Can I work from home?
P=0.33 R=1.00 RR=1.00  | What happens if I have a baby?
P=0.33 R=1.00 RR=1.00  | Do I need approval for a $30 expense?
P=0.33 R=1.00 RR=1.00  | How often do I need to change my password?
P=0.33 R=1.00 RR=1.00  | Can I share confidential information with an outside vendor?
P=0.33 R=1.00 RR=1.00  | How do I get reimbursed after a business trip?
P=0.33 R=0.50 RR=1.00  | What are the critical IT policies?
P=0.67 R=0.67 RR=1.00  | What do I need to know about working from home, keeping my laptop

Mean precision@3: 0.370
Mean recall@3:    0.907
MRR:              1.000


**⚠️ Needs your API key** — `MultiQueryRetriever` and `SelfQueryRetriever` both call the LLM internally on every `.invoke()`, so the comparison below needs a real key.

In [9]:
strategies = {
    "Plain similarity_search": lambda q: vectorstore.similarity_search(q, k=3),
    "MultiQueryRetriever": lambda q: multi_query_retriever.invoke(q),  # 1 LLM call to rewrite + up to 3x the retrieval calls, per question
    "SelfQueryRetriever": lambda q: self_query_retriever.invoke(q),    # 1 LLM call to parse a filter, then 1 filtered retrieval call
}

print(f"{'Strategy':<25} {'Precision@3':>12} {'Recall@3':>10} {'MRR':>6}")
for name, fn in strategies.items():
    _, means = evaluate_retrieval(fn, k=3)
    print(f"{name:<25} {means['precision']:>12.3f} {means['recall']:>10.3f} {means['mrr']:>6.3f}")

# Real run, actual numbers (Sarah's, 2026-07-09):
#   Plain similarity_search   0.370   0.907   1.000
#   MultiQueryRetriever       0.148   0.389   0.398   <- much WORSE, not better -- see the discussion below
#   SelfQueryRetriever        0.407   0.963   1.000   <- the clean win predicted below

Strategy                   Precision@3   Recall@3    MRR
Plain similarity_search          0.370      0.907  1.000
MultiQueryRetriever              0.148      0.389  0.398
SelfQueryRetriever               0.407      0.963  1.000


### What the real numbers say — and one of them contradicts the prediction below

```
Strategy                   Precision@3   Recall@3    MRR
Plain similarity_search          0.370      0.907  1.000
MultiQueryRetriever               0.148      0.389   0.398
SelfQueryRetriever                0.407      0.963   1.000
```

**`SelfQueryRetriever` won cleanly**, exactly as predicted — better precision, better recall, MRR still perfect. It makes one filtered `similarity_search` call; the LLM-extracted `department`/`priority` filter genuinely narrows the candidate pool before ranking, so there's no mechanism for it to do anything *but* help or tie.

**`MultiQueryRetriever` made things dramatically worse** — recall collapsed from 0.907 to 0.389, MRR from 1.000 to 0.398. That's the opposite of what the tie-back note below predicts, and it's worth actually diagnosing rather than shrugging off, because the cause isn't "multi-query retrieval is a bad idea" — it's two specific, verifiable implementation details:

1. **`include_original=False` is the default**, and cell 6 never overrides it. The retriever *only* searches using the LLM's 3 generated rephrasings — the original, usually-most-precise question text is never searched at all.
2. **The union merge has no relevance ranking.** Checked `langchain_classic`'s source directly: `_unique_documents` is `[doc for i, doc in enumerate(documents) if doc not in documents[:i]]` — a plain order-preserving dedup. The documents list is sub-query 1's results, then sub-query 2's, then sub-query 3's, concatenated. So `retrieved_ids[:3]` in `evaluate_retrieval` isn't "the best 3 across all rephrasings" — it's almost always **just sub-query 1's raw top-3, verbatim**. This harness is accidentally scoring "retrieval using only the LLM's first paraphrase," not the union `MultiQueryRetriever` is supposed to provide.

**The real-world fix**, in order of effort:
1. Set `include_original=True` — cheapest, keeps the original wording competing for a spot in the final results.
2. Don't truncate a multi-query retriever's output to the same `k` as a single-query baseline — evaluate it at the true pool size, since "return more candidates" is the entire point of the technique.
3. Add real re-ranking before truncating (Reciprocal Rank Fusion across the sub-query lists, or a cross-encoder re-ranker scored against the *original* question — the exact technique `vector_databases_for_rag` §9 already covered). LangChain's default here is closer to a demo than something to trust as-is in production.

This is exactly the point of measuring instead of assuming: the surprise wasn't "advanced isn't always better" — it's that *how a retriever is evaluated* can silently defeat its own mechanism, and that only shows up once you look at real numbers instead of the API surface.

### 🔗 Ties back to theory — this is 03's own Exercise 5, actually done (prediction below was wrong for `MultiQueryRetriever` — see the real results above)

`03_advanced_retrievers` §6 ended with "don't just trust that advanced means better — measure it." The cell above is that measurement, and the honest outcome was: `SelfQueryRetriever` won cleanly as expected, but `MultiQueryRetriever` — which had genuinely fixed a real recall failure back in 03 §2 — made *this* eval's numbers worse, for reasons that turned out to be about the retriever's default config and the eval harness's `k` truncation, not about the underlying technique being wrong. Originally this note predicted a clean recall win for `MultiQueryRetriever` too; the real run above disproved that, which is a better outcome for a notebook about measurement than if everything had gone exactly as expected — a prediction that never gets falsified isn't teaching you to measure, it's just asking you to trust a different assumption.

## 4. Beyond retrieval: is the generated answer actually good?

Precision@k/recall@k/MRR only ask "did we retrieve the right chunks?" A pipeline can retrieve the Vacation Policy perfectly and still generate an answer that invents a number, answers a different question than what was asked, or buries the actual answer in filler. That needs a different kind of check — not "was the right text present," but "did the model actually use it correctly."

In [10]:
from pydantic import BaseModel, Field


class JudgeScore(BaseModel):
    # Each Field's description IS the instruction the judge model sees -- with_structured_output
    # turns this whole class into a tool schema, so wording these precisely matters exactly
    # as much as wording a prompt would.
    faithfulness: int = Field(description="1-5: is the answer grounded ONLY in the provided context, with no invented facts?")
    relevance: int = Field(description="1-5: does the answer actually address the question that was asked?")
    reasoning: str = Field(description="One sentence justifying both scores")


def generate_rag_answer(question, k=3):
    # Deliberately the plain retriever, not multi-query/self-query -- this section is
    # about judging GENERATION quality in isolation, so retrieval is held constant/simple.
    retrieved = vectorstore.similarity_search(question, k=k)
    context = "\n".join(d.page_content for d in retrieved)
    prompt = f"Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
    answer = get_llm(max_tokens=300).invoke(prompt).content
    return answer, context


def judge_answer(question, context, answer):
    # temperature=0 (get_llm's default) here isn't optional flavor -- a judge whose
    # score changes between identical runs is useless for regression testing later.
    judge_llm = get_llm(max_tokens=300).with_structured_output(JudgeScore)
    judge_prompt = (
        f"Question: {question}\n\nContext given to the model:\n{context}\n\n"
        f"Model's answer: {answer}\n\nScore this answer."
    )
    return judge_llm.invoke(judge_prompt)  # returns a real JudgeScore object, not a string to parse


print("generate_rag_answer / judge_answer defined (no LLM call made yet).")

generate_rag_answer / judge_answer defined (no LLM call made yet).


**⚠️ Needs your API key** — the cell below makes two real LLM calls: one to generate the answer, one to judge it.

In [11]:
question = "What do I need to know about working from home, keeping my laptop secure, and expensing my home office chair?"
answer, context = generate_rag_answer(question)
print("Generated answer:\n", answer)

score = judge_answer(question, context, answer)
print("\nJudge score:", score)

Generated answer:
 Based on the provided context, here's what I can answer:

**Working from home:** You may work remotely up to 3 days per week with manager approval. Requests must be submitted at least 48 hours in advance through the HR portal.

**Keeping your laptop secure:** The context requires that all customer data must be encrypted at rest and in transit. Any suspected breach must be reported to the security team within 1 hour of discovery.

**Expensing your home office chair:** This topic is not covered in the provided context.

Judge score: faithfulness=5 relevance=3 reasoning='The answer is entirely grounded in the provided context with no invented facts, but it only partially addresses the question since it lacks information about laptop security best practices and home office chair expenses, providing incomplete guidance on two of the three topics asked.'


### 🔍 Under the hood — why the judge needs `with_structured_output`, not just a text prompt

Same mechanism as `Build_Smarter_AI_Apps` §5: `get_llm().with_structured_output(JudgeScore)` returns a `Runnable` that internally calls Claude with `JudgeScore`'s fields translated into a tool schema (via `tool_choice`, forcing a single structured tool call rather than free text), then validates and parses the result back into a real `JudgeScore` Pydantic object. Without this, you'd be regex-parsing "Faithfulness: 4/5" out of free text — fragile in exactly the way `Build_Smarter_AI_Apps` §5 already covered.

### 🔗 Ties back to theory — the same critique loop as `02_multi_agent_orchestration`'s Reflection pattern, used differently

02's Reflection section built a generate → critique → *regenerate* loop, live, inside a single request. `judge_answer` here is the same critique mechanism — an LLM checking another LLM's output against its context — but used *offline*, across a whole labeled test set, to produce a measurable quality score over time, rather than to fix one answer in the moment. Same idea, two different jobs: Reflection improves one response; an LLM-judge eval measures a system.

## 5. LangSmith — tracing, a real dataset, and regression testing

Three separate but connected pieces, all built on the same underlying tracing infrastructure:

- **Tracing** — every LLM call gets logged automatically (prompt, response, latency, token usage) the moment `LANGSMITH_TRACING=true` and a valid key are set — no `print()` statements needed to see what happened. This is true for *LangChain* calls (`get_llm()`, everywhere else in this notebook) with zero extra code; the one cell below that wraps the raw Anthropic SDK client (`wrap_anthropic`) needs that explicit step only because raw SDK calls have no LangChain callback system for the env vars to hook into.
- **Datasets** — the `eval_set` Python list from Section 2 gets uploaded once as a real, versioned, shareable LangSmith `Dataset`.
- **`evaluate()`** — runs a target function against every example in that dataset, scores each with evaluator functions (the retrieval metrics and `judge_answer` from Sections 3-4, wrapped to LangSmith's evaluator signature), and saves the whole run as a named "experiment" — re-run it after any pipeline change and diff the numbers against the previous experiment. *This* is what "regression testing" means for an LLM app: not "does the code still run," but "did the same labeled questions get measurably worse."

**⚠️ Every cell in this section needs a real LangSmith key (and Anthropic key) to run.**

### What LangSmith actually is, before any code runs

Worth pausing here, since the intro above already talks about "tracing," "datasets," and "experiments" as if you already know what those mean in this context — you don't yet, and that's the actual gap to fix first.

**LangSmith is a separate, hosted product — not a Python library that lives inside a chain.** It's built by the same company as LangChain, but the two are independent of each other: every other notebook in this curriculum used LangChain (or the raw Anthropic SDK) with zero LangSmith involved, and you could point LangSmith at an app with no LangChain in it at all — the raw-SDK `wrap_anthropic` cell just below is exactly that case. `pip install langsmith` gives you a *client library* for talking to that hosted product over HTTPS. The actual storage, the UI you'd browse, the side-by-side comparison views — none of that lives on your machine or in this notebook; it lives at smith.langchain.com, under your account.

**If you don't have an account yet:** sign up at smith.langchain.com, then grab a key from Settings → API Keys — same shape as the Anthropic key you've been entering all series (`getpass`, stored in `os.environ`, never hardcoded). The Setup cell at the top of this notebook already prompts for it and lets you skip Section 5 entirely if you leave it blank.

**The problem this exists to solve, concretely:** every notebook up to now debugged failures by reading `print()` output — `result["messages"]`, a printed `JudgeScore`, a printed dict. That's fine for one call. It stops being fine once a single `.invoke()` is secretly *several* calls — recall `create_agent`'s `agent ⇄ tools` loop from Module 01, or `MultiQueryRetriever` making one rewrite call plus up to three retrieval calls per question (Module 03). `print()` only ever shows you what you remembered to print beforehand; if a bug is hiding in the second tool call's arguments, or in exactly what a `SelfQueryRetriever` parsed a question into, you needed a print statement in that exact spot to have caught it — and `create_agent`'s internals aren't yours to add one to. LangSmith's actual value proposition: log *everything* automatically the moment tracing is switched on, with no code changes at the call site.

**The three pieces below are three separate jobs, not three views of one thing:**
- **Tracing** answers *"what actually happened on this one call?"* — a debugging tool, nothing to do with test questions or scoring.
- **Datasets** answer *"what's my set of test questions, stored somewhere that survives a kernel restart and that a teammate can also see?"* — replacing the `eval_set` Python list from Section 2.
- **`evaluate()`** answers *"did my system get better or worse?"* — runs a dataset through your pipeline, scores it, and saves the result in a form you can compare against a later run.

Each gets built up from nothing below, one piece at a time.

### What a "trace," a "run," and a "project" actually are

Three terms LangSmith uses constantly — worth defining concretely before the tracing code runs, not after:

- **A run** is one recorded unit of work: one LLM call, one tool execution, one retriever call. Every run has recorded inputs, outputs, a start/end time (so latency is just `end - start`), and — for LLM calls specifically — token usage: the exact same `input_tokens`/`output_tokens`/`cache_creation_input_tokens`/`cache_read_input_tokens` fields you've been reading out of `response_metadata` by hand since `Build_Smarter_AI_Apps` §1. LangSmith just captures and stores those automatically instead of you printing `response.usage` yourself.
- **A trace** is a *tree* of runs, not a single line in a log file. A `create_agent` call that makes two Claude calls and runs one tool produces one trace containing three runs: a root run (the overall `.invoke()`), with two child runs nested under it (the two Claude calls), and a grandchild run (the tool execution) attached under whichever Claude call requested it. Opening a trace in the LangSmith UI looks like a collapsible tree, not a scrolling log — click any single node and see *only* that node's inputs/outputs/latency, isolated from everything else that happened in the same call.
- **A project** is just a named bucket traces get filed into — `LANGSMITH_PROJECT` (set in this notebook's Setup cell, defaulting to `"module-04-eval"`) is that name. Purely organizational: a real production system typically has a `"prod"` project and a `"dev"` project so your own testing traces never get mixed in with real user traffic.

**What actually turns tracing on, mechanically — not "it's magic env vars":** every `Runnable.invoke()` call in `langchain_core` checks `langsmith.utils.tracing_is_enabled()` (which reads the `LANGSMITH_TRACING` env var) and, if it's `"true"`, auto-attaches a `LangChainTracer` callback that records that run and posts it to LangSmith's API. This is genuinely automatic, with zero code changes, for *every* LangChain call anywhere in this notebook — `get_llm()`, `generate_rag_answer`, `judge_answer`, all of it — the moment the two env vars from Setup are set. The one gap: **raw Anthropic SDK calls have no `Runnable`/callback system at all**, so there's nothing for those env vars to hook into. That's the specific, narrow reason the cell below needs `wrap_anthropic` — every other LLM call in this notebook doesn't.

In [12]:
import anthropic
from langsmith.wrappers import wrap_anthropic

# IMPORTANT: this wrapping is only needed because this cell uses the RAW anthropic
# SDK. Every get_llm()-based call elsewhere in this notebook (generate_rag_answer,
# judge_answer, and target() in the evaluate() cell below) is ALREADY traced
# automatically the moment LANGSMITH_TRACING=true and a valid key are set -- verified
# directly in langchain_core: Runnable invocation checks _tracing_v2_is_enabled(),
# which calls the exact same langsmith.utils.tracing_is_enabled() these env vars
# feed into, and auto-attaches a LangChainTracer callback. Zero wrapping needed there.
#
# Raw SDK calls have no LangChain callback system to hook into, so there's nothing
# for the env vars to auto-attach to -- that's the specific gap wrap_anthropic fills.
# wrap_anthropic returns the SAME client type back, just instrumented -- every
# .messages.create() call on traced_client logs a trace automatically.
traced_client = wrap_anthropic(anthropic.Anthropic())
response = traced_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "What is the vacation policy in one sentence?"}],
)
print(response.content[0].text)
print("\nThis call is now a trace in your LangSmith project -- check the LangSmith UI to see it.")

I don't have access to a specific vacation policy document. Could you please share the policy you're referring to, or let me know which company or organization's policy you'd like summarized?

This call is now a trace in your LangSmith project -- check the LangSmith UI to see it.


### What you'd actually see if you opened this in the LangSmith UI

There's no screenshot here, so this is the concrete walk-through instead: go to smith.langchain.com, open the `module-04-eval` project (or whatever `LANGSMITH_PROJECT` resolved to), and you'll see a list of runs, newest first — the call you just made above is at the top. Click it and you'll see the exact request that was sent (`messages=[...]`), the exact response, a latency number, and a token-usage panel — the same fields as always, just captured for you instead of printed by you.

For a `create_agent` call instead of this one flat raw call, that same view shows a *tree* instead of a single row — expand it and each `agent`/`tools` node from Module 01's graph shows up as its own clickable line, with its own inputs and outputs. That's the actual payoff: you get to see the exact tool call Claude generated and the exact raw result that came back, without having added a single print statement anywhere inside `create_agent` — which you don't have access to modify anyway, since it's LangChain's own compiled graph.

In [40]:
eval_set[-2:]

[('What are the critical IT policies?', {'it_password', 'it_security'}),
 ('What do I need to know about working from home, keeping my laptop secure, and expensing my home office chair?',
  {'finance_expense', 'hr_remote', 'it_security'})]

### What a LangSmith Dataset actually is, before the code

`eval_set` (Section 2) is a plain Python list — real and useful, but it only exists in this kernel's memory. Restart the kernel and it's gone; hand this notebook to a teammate and they get their own *copy*, with no way to tell later whether their copy has quietly drifted from yours. A LangSmith **Dataset** is the fix: a real, versioned, hosted table of test examples with a stable name and ID that anyone with access to your LangSmith workspace can read — and, for Section 6 below, that `evaluate()` can run your pipeline against directly, without you writing the `for question, relevant in eval_set:` loop yourself.

**Every example takes a fixed shape, worth knowing before it shows up in code:** `{"inputs": {...}, "outputs": {...}}` — always exactly these two keys, never renamed. `"inputs"` is whatever your system-under-test actually needs to run — here, just `{"question": "..."}`, since that's the only argument `generate_rag_answer` takes. `"outputs"` is the *ground truth* — here, `{"relevant_doc_ids": [...]}`, the correct doc IDs from `eval_set`'s second tuple element. Neither key is optional: `evaluate()` is written generically, expecting exactly this shape for *any* task, which is what lets the same `evaluate()` function work whether you're testing a RAG pipeline, a classifier, or anything else, without it knowing anything about your specific task ahead of time.

In [15]:
from langsmith import Client

ls_client = Client()  # reads LANGSMITH_API_KEY from the environment automatically -- no key passed here

# create_dataset() isn't idempotent -- it 409s if a dataset with this name already
# exists in your workspace. Re-running this cell (e.g. after a kernel restart, or
# just re-running the notebook top to bottom) hits that immediately once the first
# run has already created it. Same non-idempotent-resource-creation shape as the
# Chroma collection_name issue fixed in cell 5 above -- check first, don't assume
# a fresh run every time.
DATASET_NAME = "module-04-policy-rag-eval1"

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Dataset '{dataset.name}' already exists -- reusing it with the examples, not re-uploading examples via read_dataset()"
          f"(create_examples isn't guarded, so calling it again would just duplicate every "
          f"row on top of what's already there).")
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Labeled eval set from 04_evaluation_for_llm_apps.ipynb",
    )
    ls_client.create_examples(
        dataset_id=dataset.id,
        # LangSmith examples are always shaped {"inputs": {...}, "outputs": {...}} --
        # "inputs" is what target() below receives, "outputs" is what evaluators see
        # as reference_outputs. This is the exact same (question, relevant_ids) data
        # as eval_set above, just reshaped into that two-key format.
        examples=[
            {"inputs": {"question": q}, "outputs": {"relevant_doc_ids": sorted(ids)}}
            for q, ids in eval_set
        ],
    )
    print(f"Uploaded {len(eval_set)} examples to dataset '{dataset.name}'.")


Dataset 'module-04-policy-rag-eval1' already exists -- reusing it with the examples, not re-uploading examples via read_dataset()(create_examples isn't guarded, so calling it again would just duplicate every row on top of what's already there).


### Walking through exactly what happens for ONE row

"Runs your evaluators against the dataset" stays vague until you see it act on one concrete example. Take just the first row — `{"inputs": {"question": "How many vacation days do I get per year?"}, "outputs": {"relevant_doc_ids": ["hr_vacation"]}}` — and trace what `evaluate()` actually does with it:

1. `evaluate()` reads that row's `"inputs"` and calls `target({"question": "How many vacation days do I get per year?"})`.
2. `target()` runs `generate_rag_answer(...)` for real — one retrieval call, one generation call — plus one more retrieval call to build `retrieved_doc_ids`, returning `{"answer": "...", "context": "...", "retrieved_doc_ids": [...]}`.
3. `evaluate()` now has three things for this row: the original `inputs`, `target()`'s return value, and the dataset row's own `"outputs"`. **Worth flagging a genuinely confusing name collision here:** the dataset's `"outputs"` key (ground truth) and `target()`'s return value are two different things that both want to be called "outputs" — every evaluator function's signature resolves this by calling `target()`'s return value `outputs` and the dataset's ground truth `reference_outputs`. Get those backwards while writing your own evaluator and you'll be comparing your pipeline's answer against itself.
4. `recall_evaluator(outputs, reference_outputs)` runs: compares `outputs["retrieved_doc_ids"]` against `reference_outputs["relevant_doc_ids"]`, returns `{"key": "recall_at_3", "score": 1.0}` (assuming real retrieval found `hr_vacation`, which it should on this single-topic question).
5. `faithfulness_evaluator(inputs, outputs)` runs: makes a *second*, separate LLM call (`judge_answer`, unchanged from Section 4) to grade the generated answer, returns `{"key": "faithfulness", "score": 1.0}` (assuming a 5/5, normalized to the same 0-1 scale as recall).
6. Both `{"key", "score"}` results, plus the row's inputs/outputs/reference, get attached to this row and uploaded.

Repeat for all 9 rows, and the *whole batch* — every row's inputs, outputs, and both scores — gets saved together as one named **experiment**. That's the actual unit you're looking at afterward in the LangSmith UI: not 9 separate results scattered around, one experiment containing 9 scored rows, comparable as a whole against a *different* experiment you run later (a different `k`, a different prompt, a different retriever) to see whether a change actually helped.

In [16]:
from langsmith import evaluate


def target(inputs: dict) -> dict:
    # This is the "system under test" -- evaluate() calls this once per dataset row,
    # passing exactly the {"question": ...} dict each example's "inputs" holds.
    answer, context = generate_rag_answer(inputs["question"])
    retrieved_ids = [d.metadata["doc_id"] for d in vectorstore.similarity_search(inputs["question"], k=3)]
    return {"answer": answer, "context": context, "retrieved_doc_ids": retrieved_ids}


def recall_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    # LangSmith evaluator signature: (outputs=what target() returned,
    # reference_outputs=the example's "outputs" from create_examples above).
    # This is recall_at_k's logic, just reshaped to that (outputs, reference_outputs) signature.
    relevant = set(reference_outputs["relevant_doc_ids"])
    retrieved = set(outputs["retrieved_doc_ids"])
    score = len(relevant & retrieved) / len(relevant) if relevant else 0.0
    return {"key": "recall_at_3", "score": score}  # "key" is the metric name shown in the LangSmith UI


def faithfulness_evaluator(inputs: dict, outputs: dict) -> dict:
    # Reuses judge_answer from Section 4 verbatim -- an evaluator function can be
    # ANY Python function with this signature, including one that itself makes an LLM call.
    judged = judge_answer(inputs["question"], outputs["context"], outputs["answer"])
    return {"key": "faithfulness", "score": judged.faithfulness / 5}  # normalized to 0-1 to match recall's scale


results = evaluate(
    target,
    data="module-04-policy-rag-eval1",   # dataset name from the cell above -- evaluate() looks it up by name
    evaluators=[recall_evaluator, faithfulness_evaluator],
    experiment_prefix="module-04-baseline",  # names THIS run -- change it and re-run to create a comparable second experiment
)
print("Experiment complete -- view the side-by-side scores in the LangSmith UI.")

View the evaluation results for experiment: 'module-04-baseline-ba18c1aa' at:
https://smith.langchain.com/o/d2a2676c-ef19-4971-aed1-7972669e36cd/datasets/4a7d0675-cb50-4bf2-aa8c-5ee7894b5528/compare?selectedSessions=a00c0b9a-fc35-40af-8111-5d7cd105c50d




9it [00:21,  2.43s/it]

Experiment complete -- view the side-by-side scores in the LangSmith UI.


### 🔗 Ties back to theory — the dataset is the same `eval_set`, just no longer a local Python list

Nothing about precision@k/recall@k/MRR or `judge_answer` changed to make any of this work — `recall_evaluator` and `faithfulness_evaluator` are thin wrappers around the exact same functions from Sections 3-4, reshaped to LangSmith's `(inputs, outputs, reference_outputs) -> {"key", "score"}` signature walked through above. LangSmith's contribution here is purely *infrastructure* — versioned storage, a UI, comparability across runs — not a different evaluation methodology than Sections 3-4 already used.

Run `evaluate(...)` again after changing `k`, swapping in `MultiQueryRetriever`, or editing the RAG prompt, under a new `experiment_prefix`, and the LangSmith UI diffs the two experiments' scores side by side, per row. That diff view is the actual regression test — not a pass/fail assertion, a *comparison* you read yourself.

## 6. Optional — the raw Anthropic SDK equivalent of the judge

Same reasoning as 01/03: nothing about `with_structured_output` is vendor-specific — it's forced tool-use underneath. Here's the judge without the LangChain wrapper.

**⚠️ Needs your API key.**

In [ ]:
judge_tool = {
    "name": "score_answer",
    "description": "Score a RAG answer for faithfulness to its context and relevance to the question.",
    "input_schema": {
        "type": "object",
        "properties": {
            "faithfulness": {"type": "integer", "description": "1-5: grounded only in the given context, no invented facts"},
            "relevance": {"type": "integer", "description": "1-5: actually addresses the question asked"},
            "reasoning": {"type": "string", "description": "One sentence justifying both scores"},
        },
        "required": ["faithfulness", "relevance", "reasoning"],
    },
}

raw_client = anthropic.Anthropic()
raw_response = raw_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    tools=[judge_tool],
    # tool_choice forces this specific tool call every time -- without it, Claude could
    # legally just answer in plain text instead of calling the tool at all.
    tool_choice={"type": "tool", "name": "score_answer"},
    messages=[{
        "role": "user",
        "content": f"Question: {question}\n\nContext: {context}\n\nAnswer: {answer}\n\nScore this answer.",
    }],
)
print(raw_response.content[0].input)  # already a plain dict -- {"faithfulness": ..., "relevance": ..., "reasoning": ...}

| LangChain piece | Raw SDK equivalent |
|---|---|
| `llm.with_structured_output(JudgeScore)` | `tools=[judge_tool], tool_choice={"type": "tool", "name": "score_answer"}` |
| `JudgeScore` Pydantic model | The `input_schema` dict, hand-written |
| Parsed `JudgeScore` object returned | `response.content[0].input` — already a plain dict, no parsing needed |

Structurally identical to `Build_Smarter_AI_Apps` §5's finding: `with_structured_output` is real, saved effort (schema generation from a Pydantic model, automatic parsing back into that model) — not a different mechanism than what Claude's API already does natively.

## 7. Which layer actually matters, and when

- **Retrieval metrics** are cheap (no LLM call in the metric itself) and answer a narrow question well — use them any time you change chunking, embeddings, `k`, or swap retrievers.
- **LLM-as-judge** costs an extra LLM call per graded answer and is itself imperfect (a judge model can be wrong) — use it to catch generation-quality regressions that retrieval metrics are structurally blind to, not as a substitute for retrieval metrics.
- **LangSmith** doesn't change what you're measuring — it changes whether you can *compare* measurements over time and across teammates instead of re-reading `print()` output by eye.
- **LangSmith Engine** (Section-intro note) is the natural next step once all three of the above exist and a real agent is in production generating real traces — not something to reach for on a notebook-sized demo.

## Exercises

1. **Break something on purpose.** Shrink `chunk_size`/embedding `k`, or swap `claude-haiku-4-5` for a smaller `max_tokens` in `generate_rag_answer`, then re-run Section 3 and 4 — confirm the metrics actually move, not just the qualitative "seems worse."
2. **Add a groundedness-breaking test case.** Add a question to `eval_set` whose correct answer legitimately isn't in the knowledge base (e.g. "What's the company's parking policy?") and add an evaluator that checks the model actually says it doesn't know rather than inventing one — ties back to `Summarize_Private_Documents_RAG`'s hallucination-prevention prompt.
3. **Run a second LangSmith experiment.** Change `k` from 3 to 2 in `target`, re-run `evaluate(...)` with a new `experiment_prefix`, and compare the two experiments in the LangSmith UI — this is the regression-testing workflow for real.
4. **Add a third evaluator.** Write a `relevance_evaluator` mirroring `faithfulness_evaluator` but reading `judged.relevance`, and pass it into `evaluate(...)` alongside the other two.
5. **Try `num_repetitions=3`** on `evaluate(...)` — since `get_llm()` isn't fully deterministic even at `temperature=0` across model versions, this surfaces how much your scores vary run-to-run versus how much they move from a real pipeline change.
6. **Fix `MultiQueryRetriever` for real, using the diagnosis above.** Rebuild it with `include_original=True` and re-run the Section 3 comparison — confirm recall and MRR recover. Then go further: rewrite `strategies["MultiQueryRetriever"]` to evaluate against the *full* unique document pool (not `retrieved_ids[:3]`) and see how much of the gap was the `k`-truncation bug versus the missing original query.